# AeroIntel — 03 Evaluate + Export (Colab)

Implements spec **A6 steps 6–7**: per-class evaluation on the held-out test split, CPU latency benchmark, ONNX export, and the draft of `docs/metrics.md`.

Run this only after notebook 02 prints **ALL EPOCHS DONE** (early stopping may finish sooner — that is fine, use whatever `best.pt` exists).

In [ ]:
# @title 1. Mount Drive + install
from google.colab import drive
drive.mount('/content/drive')

%pip install -q ultralytics onnx onnxruntime
import ultralytics
ultralytics.checks()
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AeroIntel')
RUNS_DIR   = DRIVE_ROOT / 'runs'
MODELS_DIR = DRIVE_ROOT / 'models'
LOGS_DIR   = DRIVE_ROOT / 'logs'
MODELS_DIR.mkdir(exist_ok=True); LOGS_DIR.mkdir(exist_ok=True)

In [ ]:
# @title 2. Config
RUN_NAME   = 'aerointel_v1_yolo11s'   # @param {type:"string"}
VERSION    = 1                        # @param {type:"integer"}  -> models/aerointel_v1.onnx

BEST = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
assert BEST.exists(), f'{BEST} not found — sync notebook 02 cell 6 to Drive first'
print('Using best weights:', BEST)

In [ ]:
# @title 3. Restore run + dataset locally
import shutil, zipfile
from pathlib import Path

WORK_RUNS = Path('/content/runs')
local_run = WORK_RUNS / RUN_NAME
if not local_run.exists():
    shutil.copytree(RUNS_DIR / RUN_NAME, local_run)

DATA_YAML = '/content/aerointel/data.yaml'
if not Path(DATA_YAML).exists():
    with zipfile.ZipFile(DRIVE_ROOT / 'datasets/aerointel_dataset_v1_colab.zip') as z:
        z.extractall('/content/aerointel')
print('Run + dataset restored. data.yaml:', DATA_YAML)

In [ ]:
# @title 4. Per-class evaluation on the TEST split (spec A6 step 6)
# splits: run once with SPLIT='test' (the honest number), optionally again with 'valid'
SPLIT = 'test'   # @param ['test', 'valid']
from ultralytics import YOLO
import json

model = YOLO(str(BEST))
metrics = model.val(data=DATA_YAML, split=SPLIT, imgsz=640, batch=16)

names = model.names
per_class = {}
for i, name in names.items():
    per_class[name] = {
        'precision': float(metrics.box.p[i]) if i < len(metrics.box.p) else None,
        'recall':    float(metrics.box.r[i]) if i < len(metrics.box.r) else None,
        'mAP50':     float(metrics.box.ap50[i]) if i < len(metrics.box.ap50) else None,
        'mAP50-95':  float(metrics.box.ap[i].mean()) if i < len(metrics.box.ap) else None,
    }
overall = {
    'precision': float(metrics.box.mp),
    'recall':    float(metrics.box.mr),
    'mAP50':     float(metrics.box.map50),
    'mAP50-95':  float(metrics.box.map),
}
print('\n=== OVERALL ==='); print(json.dumps(overall, indent=2))
print('\n=== PER CLASS ===');  print(json.dumps(per_class, indent=2))
print('\nTarget check (A6): overall mAP50 >= 0.60 ->', 'PASS' if overall['mAP50'] >= 0.60 else 'NOT YET — iterate data/model, report honestly')
print('Classes below usable level -> ship as experimental per A6 (never hide behind the average).')

val_json = {'run': RUN_NAME, 'split': SPLIT, 'overall': overall, 'per_class': per_class}
(LOGS_DIR / f'eval_{RUN_NAME}_{SPLIT}.json').write_text(json.dumps(val_json, indent=2))
print('\nSaved:', LOGS_DIR / f'eval_{RUN_NAME}_{SPLIT}.json')
print('Confusion matrix + PR curves are in', local_run)

In [ ]:
# @title 5. CPU latency benchmark (A6 target: < 1 s per image on laptop-class CPU)
import time, statistics, glob
from ultralytics import YOLO

test_imgs = sorted(glob.glob('/content/aerointel/test/images/*'))[:20]
if not test_imgs:
    test_imgs = sorted(glob.glob('/content/aerointel/valid/images/*'))[:20]
assert test_imgs, 'No images found for the latency test'

cpu_model = YOLO(str(BEST))
_ = cpu_model.predict(test_imgs[0], device='cpu', imgsz=640, verbose=False)  # warmup
times = []
for p in test_imgs * 3:
    t0 = time.perf_counter()
    cpu_model.predict(p, device='cpu', imgsz=640, conf=0.40, iou=0.50, verbose=False)
    times.append((time.perf_counter() - t0) * 1000)
latency = {'device': 'colab cpu (proxy for laptop cpu)', 'n_images': len(test_imgs) * 3,
           'p50_ms': round(statistics.median(times), 1),
           'p95_ms': round(sorted(times)[int(0.95 * len(times)) - 1], 1),
           'conf': 0.40, 'iou': 0.50}
print(json.dumps(latency, indent=2))
print('\nA6 target < 1000 ms: ', 'PASS' if latency['p95_ms'] < 1000 else 'REVIEW (resize/ONNX will help; re-measure on the real laptop too)')
(LOGS_DIR / f'latency_{RUN_NAME}.json').write_text(json.dumps(latency, indent=2))

In [ ]:
# @title 6. Export ONNX + record model version (spec A6 step 7)
from ultralytics import YOLO
import shutil, datetime

model = YOLO(str(BEST))
model.export(format='onnx', imgsz=640, opset=12, simplify=True)
produced = BEST.with_suffix('.onnx')
if not produced.exists():
    produced = BEST.parent / 'best.onnx'
assert produced.exists(), 'ONNX export did not produce a file where expected'

onnx_name = f'aerointel_v{VERSION}.onnx'
shutil.copy2(produced, MODELS_DIR / onnx_name)

registry = {
    'name': f'aerointel_v{VERSION}',
    'type': 'yolo11-onnx',
    'path': f'models/{onnx_name}',
    'classes': {'0': 'crack', '1': 'corrosion', '2': 'dent', '3': 'missing_fastener'},
    'version': f'v{VERSION}',
    'imgsz': 640,
    'trained_from': RUN_NAME,
    'exported_at': datetime.datetime.now().isoformat(),
}
(MODELS_DIR / 'registry.json').write_text(json.dumps(registry, indent=2))
print('Exported ->', MODELS_DIR / onnx_name)
print('Registry ->', MODELS_DIR / 'registry.json')
print('\nNext: download it from Drive into your repo models/ folder (do NOT commit big files — see B4).')

In [ ]:
# @title 7. Draft docs/metrics.md (fill placeholders honestly — A16 acceptance #6)
import json

eval_json = json.loads((LOGS_DIR / f'eval_{RUN_NAME}_test.json').read_text())
lat_json  = json.loads((LOGS_DIR / f'latency_{RUN_NAME}.json').read_text())

o = eval_json['overall']
lines = []
lines.append(f'# AeroIntel model metrics — {RUN_NAME} (draft)')
lines.append('')
lines.append('## Overall (held-out test split)')
lines.append(f"- precision: {o['precision']:.3f} | recall: {o['recall']:.3f} | mAP50: {o['mAP50']:.3f} | mAP50-95: {o['mAP50-95']:.3f}")
lines.append('')
lines.append('## Per class (never hide a weak class behind the average)')
for name, m in eval_json['per_class'].items():
    lines.append(f"- **{name}**: P={m['precision']:.3f} R={m['recall']:.3f} mAP50={m['mAP50']:.3f} mAP50-95={m['mAP50-95']:.3f}")
lines += [
    '',
    '## Latency (CPU)',
    f"- p50: {lat_json['p50_ms']} ms | p95: {lat_json['p95_ms']} ms (conf 0.40, IoU 0.50, imgsz 640) — re-measure on the demo laptop before final submission.",
    '',
    '## Field test set (spec A6 — fill in after collecting >=50 unseen images)',
    '- TODO: same table on the field set. Expect a drop vs the dataset metrics; state it honestly.',
    '',
    '## Known weaknesses / experimental classes',
    '- TODO: list classes below usable level and mark them experimental per A6.',
]
md = '\n'.join(lines) + '\n'
out = DRIVE_ROOT / 'metrics_draft.md'
out.write_text(md)
print(md)
print('Saved -> Drive AeroIntel/metrics_draft.md — copy into docs/metrics.md')